# Seaborn

COMP 6934 \
Karteek Popuri

In this notebook, we will introduce the Seaborn library. Seaborn provides a high-level interface for creating Matplotlib plots. We will study the basics of Seaborn through line and scatter plots, specifically,

- ``lineplot()``
- ``scatterplot()``


Seaborn has excellent API documentation and tutorials: https://seaborn.pydata.org/


## Import Libraries

Seaborn is commonly loaded with the name ``sns``.

Why ``sns``? It started as an inside joke during initial development, referring to the character Samuel Norman Seaborn from The West Wing. Lesson to be learned: carefully choose names as they are hard to change once established.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

## Load Data

In [ ]:
df = pd.read_csv('CA_videos.csv', parse_dates=['trending_date', 'publish_time'])

Extract the month from the trending date information (which was parsed as datetime when the data was initially loaded).

In [ ]:
df['trending_month'] = df.trending_date.dt.month

We will just use just the top 4 channels based on the highest number of trending videos. First, we need to determine what are those channels.

In [ ]:
top4_channels = df[['video_id', 'channel_title']].groupby('channel_title').nunique().reset_index().sort_values(by='video_id', ascending=False).head(4).channel_title

In [ ]:
top4_channels

In [ ]:
df = df[df.channel_title.isin(top4_channels)]

In [ ]:
df.info()

In [ ]:
df.head(50)

## Line Plots

Let's make a line plot to address the question: **What are the average number of views per month for our top channels?**

### Matplotlib Implementation

Let's first create our plot of average views per month using Matplotlib.

We need to calculate the data for the plot by using `groupby()` to calculate the average views per month per channel. (Note that the 'per month' and 'per channel' tell us which columns to use in our groupby.)

In [ ]:
df_m = df[['channel_title', 'trending_month', 'views']].groupby(['channel_title', 'trending_month']).mean().reset_index()

In [ ]:
df_m

In [ ]:
fig, ax = plt.subplots()

for channel in top4_channels:
    df_tmp = df_m[df_m.channel_title == channel]
    ax.plot(df_tmp.trending_month, df_tmp.views, label=channel)
    
ax.legend()
ax.set_xlabel('Trending Month')
ax.set_ylabel('views')

### Seaborn implementation

Now let's plot the same data using Seaborn's ``lineplot()`` function.

In [ ]:
sns.lineplot(x='trending_month', y='views', hue='channel_title', data=df)

We have created the same plot using one line of code.

In fact, the plot is even better than the Matplotlib plot because it has included shading to represent the variance in number of views. 

In this case, the shading represents the 95% confidence interval for the average.

Also, we did not even have to calcualte the averages ourselves using groupby -- Seaborn calculated the averages for us. And it has included shading for the confidence intervals. It would have added a lot of extra work to calculate the confidence intervals ourselves and manually draw the shading using Matplotlib.

## Seaborn Functionality

One important difference from Matplotlib is that the pandas DataFrame can be passed directly to Seaborn. Specifiying the x and y data requires only providing the column names that should be used for those axes. (Also the colour, via `hue=`.)

Let's break down the parameters of the function call:
- `x=` specifies the x-data. This can be a column of a pandas DataFrame or an array of data.
- `y=` specifies the y-data.
- `data=` specifies a pandas DataFrame. Use this if you want other arguments to reference data in the DataFrame.
- `hue=` is the grouping variable to produce lines of different colours.

Seaborn is built upon Matplotlib and integrates strongly with it. There are two important features to highlight.

**1. Seaborn functions return a Matplotlib Axes object.** All the regular Matplotlib functionality can then be used with that Axes.

In [ ]:
ax = sns.lineplot(x='trending_month', y='views', data=df, hue='channel_title', legend=False)

ax.set_xlabel('Trending Month')
ax.set_title('Top 4 Trending YouTube Channels')
ax.set_yscale('log')

**2. Seaborn can also draw onto existing Axes objects.** Just pass your Axes object to the function using the ``ax=`` parameter.

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

sns.lineplot(x=df['trending_month'], y='views', data=df, hue=df['channel_title'], ax=ax)

ax.set_xlabel('Trending Month')
ax.set_title('Top 4 Trending YouTube Channels')
# ax.set_yscale('log')

Note that the pandas DataFrame was directly passed to the Seaborn function call. The x and y data were specified by the column name of the DataFrame, same as with the column that the colour and markers were based on. Seaborn works well with tidy data.

Instead of the semantic mapping above, arrays of data could be passed for each argument instead. (In this case, the pandas Series for each data.)

In [ ]:
df.trending_month

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

sns.lineplot(x=df.trending_month, y=df.views, hue=df.channel_title, ax=ax)  # arrays passed directly to arguments

ax.set_xlabel('Trending Month')
ax.set_title('Top 4 Trending YouTube Channels')
ax.set_yscale('log')

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

sns.lineplot(x=df.trending_month, y='views', data=df, hue=df.channel_title, ax=ax)  # mix of arrays and data keys

ax.set_xlabel('Trending Month')
ax.set_title('Top 4 Trending YouTube Channels')
ax.set_yscale('log')

Notice that the pandas DataFrame was directly passed to the Seaborn function call. The x and y data were specified by the column name of the DataFrame. The colours and marker styles also reference the column to use for their data.

## Scatter Plots

Seaborn includes its own `scatterplot()` function that has many of the same benefits as lineplot.

Let's make a scatter plot of the number of comments vs number of views for all the videos for the top 4 channels.

In [ ]:
fig, ax = plt.subplots()

sns.scatterplot(x='views', y='comment_count', data=df, hue='channel_title', ax=ax)

We can add marker styles based on channel title using the ``style=`` parameter.

In [ ]:
fig, ax = plt.subplots()

ax = sns.scatterplot(x='views', y='likes', data=df, hue='channel_title', style='channel_title', ax=ax)

ax.set_title("Top 4 Trending YouTube Channels")
ax.set_xscale('log')
ax.set_yscale('log')

# Summary

Seaborn offers a high-level interface to Matplotlib. It still allows you to do all the fine-grain tuning in Matplotlib that you have learned how to do, but permits you to skip many of the common, tedious tasks. It also creates plots with a nice aesthetic, and can do many transformations for you.

The documentation for Seaborn's API is excellent: https://seaborn.pydata.org/api.html

There are also excellent tutorials on the Seaborn website.